<a href="https://colab.research.google.com/github/J4SIB/ai-course-gp/blob/main/text_to_speech.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gTTS
!pip install --upgrade gradio

  Using cached click-8.1.8-py3-none-any.whl.metadata (2.3 kB)
Using cached click-8.1.8-py3-none-any.whl (98 kB)
  Attempting uninstall: click
    Found existing installation: click 8.3.3
    Uninstalling click-8.3.3:
      Successfully uninstalled click-8.3.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
typer 0.24.1 requires click>=8.2.1, but you have click 8.1.8 which is incompatible.
  Using cached click-8.3.3-py3-none-any.whl.metadata (2.6 kB)
Using cached click-8.3.3-py3-none-any.whl (110 kB)
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you 

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display


In [ ]:
text = "Cześć witam na lekcji o generatorach mowy pisanych w JĘZYKU WĘŻA"
tssl = gTTS(text,lang="pl")
tssl.save("powitanie.mp3")
print("Odtwarzanie: powitanie")
display(Audio("powitanie.mp3"))

Odtwarzanie: powitanie


In [ ]:
slownik = {
    'start': 'Zaczynajmy grę.',
    "dobrze": "Gratulacje twoja odpowiedz jest prawidłowa.",
    'źle': "Niestety nieprawidłowo. Spróbuj jeszcze raz.",
    'Koniec': 'Koniec gry.',
}

print("Odtwarzanie: komunikaty gry")
for klucz, fraza in slownik.items():
  tts=gTTS(fraza,lang='pl')
  filename = f'{klucz}.mp3'
  tts.save(filename)
  print(f'{klucz.upper()}: {fraza}')
  display(Audio(filename))

Odtwarzanie: komunikaty gry
START: Zaczynajmy grę.


DOBRZE: Gratulacje twoja odpowiedz jest prawidłowa.


ŹLE: Niestety nieprawidłowo. Spróbuj jeszcze raz.


KONIEC: Koniec gry.


In [ ]:
from pydub import AudioSegment
from pydub.playback import play
from IPython.display import Audio

def change_pitch(audio_segment, semitones=0):
  new_sample_rate=int(audio_segment.frame_rate * (2 ** (semitones / 12)))
  return audio_segment._spawn(audio_segment.raw_data, overrides={"frame_rate": new_sample_rate}).set_frame_rate(audio_segment.frame_rate)

def change_speed(audio_segment, speed=1.0):
  return audio_segment._spawn(audio_segment.raw_data, overrides={'frame_rate': int(audio_segment.frame_rate * speed)}).set_frame_rate(audio_segment.frame_rate)

sound = AudioSegment.from_file("powitanie.mp3")
sound_fast = change_speed(sound, 1.5)
sound_low_pitch = change_pitch(sound, -4)
sound_high_pitch = change_pitch(sound, 4)

sound_fast.export('szybciej.mp3', format="mp3")
sound_low_pitch.export('niżej.mp3', format="mp3")
sound_high_pitch.export('wyżej.mp3', format="mp3")

print("szybszy głos:")
display(Audio('szybciej.mp3'))

print("podniesiony pitch:")
display(Audio('wyżej.mp3'))

print("obniżony pitch")
display(Audio('niżej.mp3'))

szybszy głos:


podniesiony pitch:


obniżony pitch


In [3]:
import gradio as gr
from gtts import gTTS
from pydub import AudioSegment
import tempfile
import os

# Funkcja do modyfikacji prędkości
def change_speed(audio_segment, speed=1.0):
    new_frame_rate = int(audio_segment.frame_rate * speed)
    return audio_segment._spawn(audio_segment.raw_data, overrides={'frame_rate': new_frame_rate}).set_frame_rate(audio_segment.frame_rate)

# Funkcja do modyfikacji wysokości
def change_pitch(audio_segment, semitones=0):
    new_sample_rate = int(audio_segment.frame_rate * (2 ** (semitones / 12)))
    return audio_segment._spawn(audio_segment.raw_data, overrides={'frame_rate': new_sample_rate}).set_frame_rate(audio_segment.frame_rate)

# Funkcja główna
def tts_with_effects(text, speed, pitch):
    # 1. Generowanie mowy
    tts = gTTS(text=text, lang='pl')
    temp_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    tts.save(temp_file.name)

    # 2. Wczytanie wygenerowanego audio
    audio = AudioSegment.from_file(temp_file.name, format="mp3")

    # 3. Zmiana wysokosci
    audio = change_pitch(audio, pitch)

    # 4. Zmiana prędkości
    audio = change_speed(audio, speed)

    # 5. Zapis do pliku końcowego
    output_file = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
    audio.export(output_file.name, format="mp3")

    # 6. Zwrócenie ścieżki audio
    return output_file.name

#Interfejs Gradio
with gr.Blocks() as demo:
    gr.Markdown("## Generator mowy")
    with gr.Row():
        text_input = gr.Textbox(label="Tekst do przeczytania", placeholder="Wpisz tutaj tekst...")
    with gr.Row():
        speed_input = gr.Slider(0.5, 2.0, value=1.0, step=0.1, label="Prędkość mowy (0.5 = wolniej, 2.0 = szybciej)")
        pitch_input = gr.Slider(-12, 12, value=0, step=1, label="Zmiana wysokości głosu w półtonach (-12 do +12)")
    with gr.Row():
        output_audio = gr.Audio(label="Wygenerowana mowa", type="filepath")
    with gr.Row():
        generate_btn = gr.Button("Generuj mowę")

    generate_btn.click(tts_with_effects, inputs=[text_input, speed_input, pitch_input], outputs=output_audio)


demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1fcd46922ecaece7b7.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
